# 02. Базовые методы: пороговый отбор и логистическая регрессия (+ превью XGBoost)

**Цель:** получить первые строки таблицы результатов по единому протоколу и убедиться,
что конвейер «данные → модель → значимость» работает от начала до конца.

**Протокол (одинаковый для всех методов в работе):**
1. Разбиение 60 / 20 / 20 (train / val / test), стратифицированное; веса в каждой части
   пересчитаны так, чтобы суммы s и b совпадали с полным набором (`src/data.py`).
2. Порог классификатора выбирается по максимуму AMS на **val**.
3. Итоговые числа (AMS, Z0, s, b) считаются на **test** при зафиксированном пороге — один раз.
4. Сохраняем результаты в `results/baseline_results.csv`.

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Корень репозитория: поднимаемся, пока не найдём каталог src/
ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data import (load_higgsml, find_data_file, FEATURES, DER_FEATURES, PRI_FEATURES,
                      FEATURE_LABELS, MISSING, with_nan, missing_mask, expected_yields,
                      split_train_val_test)
from src.metrics import (ams, z0, weighted_counts, weighted_roc, significance_scan, best_threshold,
                         evaluate_at_threshold, ks_overtraining, balanced_weights)
from src.plots import setup_style, hist_sig_bkg, grid_all_features, plot_roc, plot_significance_scan, plot_classifier_output

setup_style()
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)
DATA_PATH = find_data_file(ROOT / "data")
print("Файл данных:", DATA_PATH.name)
if "synthetic" in DATA_PATH.name:
    print("ВНИМАНИЕ: это синтетические данные для проверки кода, а не физика!")

In [ ]:
df = load_higgsml(DATA_PATH)
train, val, test = split_train_val_test(df, test_size=0.2, val_size=0.2, seed=42)
for name, part in (("train", train), ("val", val), ("test", test)):
    yl = expected_yields(part, "w")
    print(f"{name:5s}: строк {len(part):>8,}   s = {yl['s']:8.1f}   b = {yl['b']:12.1f}")

results = []   # сюда складываем строки таблицы результатов
roc_curves = {}
scans = {}

## 1. Пороговый отбор (cut-based)

Классический анализ: несколько порогов по самым разделяющим признакам.
Сканируем сетку порогов на **val**, лучший набор применяем к **test**.
Признаки выбраны по результатам `01_eda` (одиночный AUC): окно по m_MMC,
верхний порог на m_T(MET, ℓ), верхний порог на ΔR(τ, ℓ).

In [ ]:
def cut_selection(part, m_lo, m_hi, mt_max, dr_max):
    m = part["DER_mass_MMC"].to_numpy()
    sel = (m != MISSING) & (m >= m_lo) & (m <= m_hi)
    sel &= part["DER_mass_transverse_met_lep"].to_numpy() <= mt_max
    sel &= part["DER_deltar_tau_lep"].to_numpy() <= dr_max
    return sel

grid = [(m_lo, m_hi, mt, dr)
        for m_lo in (95, 100, 105, 110)
        for m_hi in (135, 140, 150, 160)
        for mt in (40, 60, 80, 1e9)
        for dr in (2.5, 3.0, 3.5, 1e9)]

best_cut, best_ams = None, -1
for params in grid:
    sel = cut_selection(val, *params)
    s, b = weighted_counts(val["y"], sel, val["w"])
    a = float(ams(s, b))
    if a > best_ams:
        best_cut, best_ams = params, a
print("Лучшие пороги на val:", best_cut, f"AMS_val = {best_ams:.3f}")

sel_test = cut_selection(test, *best_cut)
s, b = weighted_counts(test["y"], sel_test, test["w"])
results.append({"метод": "пороговый отбор", "AUC": np.nan, "AMS_val": best_ams,
                "AMS_test": float(ams(s, b)), "Z0_test": float(z0(s, b)), "s_test": s, "b_test": b})
pd.DataFrame(results).round(3)

## 2. Логистическая регрессия

Линейная модель — ориентир «сколько даёт просто многомерность». Для неё нужны:
- замена −999 на NaN → импутация медианой + бинарные флаги «было −999»
  (`SimpleImputer(add_indicator=True)`);
- стандартизация признаков;
- веса: при обучении уравниваем суммарный вклад классов (иначе модель почти не видит сигнал),
  средний вес = 1 (`src.metrics.balanced_weights`), а значимость считаем с настоящими весами `w`.

Порог выбирается по максимуму AMS на val, но не на «хвосте» из нескольких событий:
`best_threshold(..., min_events=200)` — иначе порог не переносится на test.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X_train, X_val, X_test = (with_nan(p)[FEATURES].to_numpy() for p in (train, val, test))
y_train, y_val, y_test = (p["y"].to_numpy() for p in (train, val, test))
w_train, w_val, w_test = (p["w"].to_numpy() for p in (train, val, test))
wb_train, wb_val = balanced_weights(y_train, w_train), balanced_weights(y_val, w_val)

logreg = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=2000),
)
logreg.fit(X_train, y_train, logisticregression__sample_weight=wb_train)

score_val = logreg.predict_proba(X_val)[:, 1]
score_test = logreg.predict_proba(X_test)[:, 1]

bt = best_threshold(y_val, score_val, w_val, metric="ams")
ev = evaluate_at_threshold(y_test, score_test, w_test, bt["threshold"])
_, _, auc_val = weighted_roc(y_val, score_val, w_val)
roc_curves["логистическая регрессия"] = weighted_roc(y_test, score_test, w_test)
scans["логистическая регрессия"] = significance_scan(y_val, score_val, w_val)
results.append({"метод": "логистическая регрессия", "AUC": auc_val, "AMS_val": bt["significance"],
                "AMS_test": ev["ams"], "Z0_test": ev["z0"], "s_test": ev["s"], "b_test": ev["b"]})
print(f"порог = {bt['threshold']:.3f}, отобрано на val {bt['n_selected']} событий")
pd.DataFrame(results).round(3)

## 3. Превью: градиентный бустинг (XGBoost) с параметрами по умолчанию

Полноценная настройка — в `03_bdt.ipynb`. Здесь лишь показываем, что нелинейная модель
даёт заметный прирост при тех же данных. XGBoost работает с NaN напрямую.
Если есть GPU (NVIDIA), раскомментируйте `device="cuda"`.

In [ ]:
import xgboost as xgb

bdt = xgb.XGBClassifier(
    n_estimators=400, learning_rate=0.1, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss", early_stopping_rounds=30,
    random_state=42, n_jobs=-1,
    # device="cuda",
)
bdt.fit(X_train, y_train, sample_weight=wb_train,
        eval_set=[(X_val, y_val)], sample_weight_eval_set=[wb_val], verbose=False)
print("деревьев после ранней остановки:", bdt.best_iteration + 1)

score_train_b = bdt.predict_proba(X_train)[:, 1]
score_val_b = bdt.predict_proba(X_val)[:, 1]
score_test_b = bdt.predict_proba(X_test)[:, 1]

bt_b = best_threshold(y_val, score_val_b, w_val, metric="ams")
ev_b = evaluate_at_threshold(y_test, score_test_b, w_test, bt_b["threshold"])
_, _, auc_val_b = weighted_roc(y_val, score_val_b, w_val)
roc_curves["XGBoost (по умолчанию)"] = weighted_roc(y_test, score_test_b, w_test)
scans["XGBoost (по умолчанию)"] = significance_scan(y_val, score_val_b, w_val)
results.append({"метод": "XGBoost (по умолчанию)", "AUC": auc_val_b, "AMS_val": bt_b["significance"],
                "AMS_test": ev_b["ams"], "Z0_test": ev_b["z0"], "s_test": ev_b["s"], "b_test": ev_b["b"]})

ks = ks_overtraining(score_train_b, y_train, score_test_b, y_test)
print("KS-тест train/test:", {k: round(v["pvalue"], 3) for k, v in ks.items()},
      "(p > 0.05 — переобучения не видно)")
pd.DataFrame(results).round(3)

## 4. Графики для главы 3

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
plot_roc(roc_curves, ax=axes[0]); axes[0].set_title("ROC на test")
plot_significance_scan(scans, ax=axes[1]); axes[1].set_title("AMS от порога (val)")
plot_classifier_output(score_train_b, y_train, w_train, score_test_b, y_test, w_test, ax=axes[2])
axes[2].set_title("выход XGBoost: train vs test")
fig.tight_layout()
fig.savefig(RESULTS / "fig_baseline.png", bbox_inches="tight")
plt.show()

## 5. Таблица результатов

Это первая версия главной таблицы работы. В `03_bdt.ipynb` и `04_mlp.ipynb` к ней добавятся
настроенные модели и разброс по 5 разбиениям (среднее ± σ).

In [ ]:
table = pd.DataFrame(results)
table["прирост AMS к порогам, %"] = 100 * (table["AMS_test"] / table.loc[0, "AMS_test"] - 1)
table.to_csv(RESULTS / "baseline_results.csv", index=False)
display(table.round(3))
print("Сохранено:", RESULTS / "baseline_results.csv")

## Что дальше

- `03_bdt.ipynb`: подбор гиперпараметров XGBoost/LightGBM (Optuna, 30–50 проб), 5 повторов
  с разными seed, кривые обучения.
- `04_mlp.ipynb`: многослойный перцептрон в PyTorch на GPU при том же протоколе.
- Сравнение с лидербордом Kaggle: обучиться на `KaggleSet == "t"`, посчитать AMS на `"v"`
  с весами `KaggleWeight` (победитель 2014 г. — AMS ≈ 3.8).